# Multi-Source Pipeline Health: Ingestion, Conformance & Drift Monitoring
**Author:** Bawelile Gule · Data Scientist  
**Stack:** PySpark · Spark SQL · Delta Lake · Data Quality · Drift Detection · Databricks-compatible  
[![Portfolio](https://img.shields.io/badge/Portfolio-Bawelile.github.io-1A5276?style=flat-square)](https://Bawelile.github.io)

---
## Business Problem

A platform that ingests data from many independent upstream sources (e.g., multiple bottler/partner systems feeding a shared analytics platform) faces a problem single-source pipelines don't: **each source can drift independently** — different schemas, different freshness, different data quality — while the *shared downstream tables* assume consistency.

If Source #7's feed silently breaks, a notebook that only checks 'overall' metrics may show nothing wrong — while predictions for Source #7 quietly degrade.

This pipeline simulates **11 independent upstream sources** feeding a shared product-availability table, and builds:
1. Per-source ingestion with schema conformance + quarantine (Bronze)
2. Freshness-aware joining across sources with different latencies (Silver)
3. A unified, model-ready table with per-source confidence flags (Gold)
4. Per-source data quality + drift monitoring — with a simulated drift injected so you can see it fire

## Architecture
```
11 source feeds (heterogeneous schema, freshness, quality)
   ↓
Bronze: per-source ingestion + validation + quarantine
   ↓
Silver: schema conformance + freshness-aware join
   ↓
Gold: unified table + per-source confidence flag
   ↓
Monitoring: per-source data quality + drift detection (data + concept)
```

**Run all cells top to bottom on Google Colab.**

---
## 0 · Setup

Colab ships with Java pre-installed but `JAVA_HOME` isn't always set, and the path varies by Colab image. This cell auto-detects it instead of hardcoding.

In [ ]:
%%capture
!pip install pyspark==3.5.0 delta-spark==3.1.0 plotly --quiet

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, glob, subprocess

# --- Auto-detect JAVA_HOME (don't hardcode — Colab's path varies) ---
if 'JAVA_HOME' not in os.environ or not os.path.exists(os.environ.get('JAVA_HOME','')):
    candidates = glob.glob('/usr/lib/jvm/*') + glob.glob('/opt/java*')
    java_home = None
    for c in candidates:
        if os.path.exists(os.path.join(c, 'bin', 'java')):
            java_home = c
            break
    if java_home is None:
        # Not found — install a JDK
        subprocess.run(['apt-get','install','-y','-qq','openjdk-11-jdk-headless'],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        candidates = glob.glob('/usr/lib/jvm/*')
        for c in candidates:
            if os.path.exists(os.path.join(c, 'bin', 'java')):
                java_home = c
                break
    os.environ['JAVA_HOME'] = java_home
    os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

print('JAVA_HOME ->', os.environ['JAVA_HOME'])

In [ ]:
import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pyspark.sql import SparkSession, functions as F, Window, types as T
from delta import configure_spark_with_delta_pip

builder = (SparkSession.builder.appName('MultiSourcePipelineHealth')
    .config('spark.sql.extensions','io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog','org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.driver.memory','4g'))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

N_SOURCES = 11
SOURCE_IDS = [f'SRC_{i:02d}' for i in range(1, N_SOURCES+1)]
PRODUCTS = ['Cola_12pk','Cola_2L','DietCola_12pk','Sparkling_8pk','Juice_6pk']
DAYS = pd.date_range('2024-10-01','2024-12-31', freq='D')

print(f'Spark {spark.version} session ready (Delta Lake configured)')
print(f'Simulating {N_SOURCES} independent upstream sources × {len(PRODUCTS)} products × {len(DAYS)} days')

---
## 1 · Simulate 11 heterogeneous source feeds

Each source has its own column naming, its own data quality profile, and its own arrival-delay pattern — deliberately inconsistent, the way independent upstream systems are in reality.

In [ ]:
np.random.seed(7)

SOURCE_PROFILES = {}
for i, src in enumerate(SOURCE_IDS):
    SOURCE_PROFILES[src] = dict(
        schema_variant = i % 3,            # 0,1,2 -> three different column-naming conventions
        null_rate      = np.random.uniform(0.0, 0.05),
        arrival_delay  = np.random.choice([0,0,0,1,2], p=[0.6,0.15,0.1,0.1,0.05]),
        avg_inventory  = np.random.randint(50, 400),
    )

# SRC_07 will be our 'problem source' for the drift demo later — flag it now, used in Section 5
DRIFT_SOURCE = 'SRC_07'
DRIFT_START  = pd.Timestamp('2024-12-15')

def make_source_frame(src, profile):
    rows = []
    for day in DAYS:
        for product in PRODUCTS:
            base_inv = profile['avg_inventory'] * (1 + 0.1*np.sin(2*np.pi*day.dayofyear/365))
            inv = max(int(base_inv + np.random.normal(0, base_inv*0.1)), 0)
            sold = max(int(inv*0.15 + np.random.normal(0, 5)), 0)
            on_hand_after = max(inv - sold, 0)
            out_of_stock = int(on_hand_after == 0)

            if np.random.random() < profile['null_rate']:
                on_hand_after = None

            # DRIFT INJECTION: from DRIFT_START, SRC_07 starts reporting on_hand in CASES not UNITS
            if src == DRIFT_SOURCE and day >= DRIFT_START and on_hand_after is not None:
                on_hand_after = max(round(on_hand_after / 12), 0)  # 12 units/case

            rows.append(dict(report_date=day, product=product,
                              on_hand=on_hand_after, units_sold=sold,
                              out_of_stock=out_of_stock))

    df = pd.DataFrame(rows)

    variant = profile['schema_variant']
    if variant == 0:
        df = df.rename(columns={'report_date':'date','product':'sku','on_hand':'inventory_units'})
    elif variant == 1:
        df = df.rename(columns={'report_date':'snapshot_dt','product':'item_code','on_hand':'qty_on_hand'})
    else:
        df = df.rename(columns={'report_date':'date','product':'product_id','on_hand':'available_units'})

    date_col = [c for c in df.columns if c in ('date','snapshot_dt')][0]
    df['arrival_ts'] = pd.to_datetime(df[date_col]) + pd.Timedelta(days=profile['arrival_delay'])
    df['source_id'] = src
    return df

SOURCE_FRAMES = {src: make_source_frame(src, prof) for src, prof in SOURCE_PROFILES.items()}

print('Schema preview — three different naming conventions across sources:')
for variant_example in ['SRC_01','SRC_02','SRC_03']:
    print(f'\n{variant_example} columns: {list(SOURCE_FRAMES[variant_example].columns)}')

print(f'\nSRC_07 (drift source) sample around drift date:')
src07 = SOURCE_FRAMES['SRC_07']
date_col_07 = [c for c in src07.columns if c in ('date','snapshot_dt')][0]
print(src07[src07[date_col_07].between('2024-12-13','2024-12-17')].head(8))

---
## 2 · Bronze — per-source ingestion with validation & quarantine

Each source lands in its OWN Bronze table (preserving native schema). A validation pass runs per source — bad records get quarantined, not silently dropped or allowed to corrupt downstream tables.

In [ ]:
def validate_source(pdf, src):
    """Per-source data quality checks. Returns (clean_df, quarantine_df, quality_report)."""
    df = pdf.copy()
    issues = pd.Series(False, index=df.index)
    report = {}

    qty_col = [c for c in df.columns if c in ('inventory_units','qty_on_hand','available_units')][0]
    null_mask = df[qty_col].isna()
    report['null_rate'] = float(null_mask.mean())
    issues |= null_mask

    range_mask = (df[qty_col].fillna(0) < 0) | (df[qty_col].fillna(0) > 5000)
    report['range_violation_rate'] = float(range_mask.mean())
    issues |= range_mask

    date_col = [c for c in df.columns if c in ('date','snapshot_dt')][0]
    prod_col = [c for c in df.columns if c in ('sku','item_code','product_id')][0]
    dup_mask = df.duplicated(subset=[date_col,prod_col], keep='first')
    report['duplicate_rate'] = float(dup_mask.mean())
    issues |= dup_mask

    clean = df[~issues].copy()
    quarantine = df[issues].copy()
    report['row_count'] = len(df)
    report['quarantined_rows'] = len(quarantine)
    report['quarantine_rate'] = len(quarantine)/len(df) if len(df)>0 else 0
    return clean, quarantine, report

BRONZE = {}
QUARANTINE = {}
QUALITY_REPORTS = {}
for src, pdf in SOURCE_FRAMES.items():
    clean, quarantine, report = validate_source(pdf, src)
    BRONZE[src] = clean
    QUARANTINE[src] = quarantine
    QUALITY_REPORTS[src] = report

quality_df = pd.DataFrame(QUALITY_REPORTS).T
quality_df.index.name = 'source_id'
print('Per-source Bronze validation summary:')
print(quality_df[['row_count','null_rate','range_violation_rate','duplicate_rate','quarantine_rate']].round(4))

---
## 3 · Silver — schema conformance + freshness-aware join

Map every source's schema to a canonical schema, then build a per-(date, product) view that tracks, for EACH source, how many days old its data is as of a given 'as-of' date — because not every source reports on the same day.

In [ ]:
CANONICAL_COLS = ['report_date','product','source_id','inventory_units','units_sold','out_of_stock','arrival_ts']

def conform_schema(df, src):
    d = df.copy()
    rename_map = {}
    for c in d.columns:
        if c in ('date','snapshot_dt'): rename_map[c] = 'report_date'
        if c in ('sku','item_code','product_id'): rename_map[c] = 'product'
        if c in ('inventory_units','qty_on_hand','available_units'): rename_map[c] = 'inventory_units'
    d = d.rename(columns=rename_map)
    return d[CANONICAL_COLS]

SILVER_FRAMES = []
for src, df in BRONZE.items():
    SILVER_FRAMES.append(conform_schema(df, src))

silver_pdf = pd.concat(SILVER_FRAMES, ignore_index=True)
silver_pdf['report_date'] = pd.to_datetime(silver_pdf['report_date'])
silver_pdf['arrival_ts']  = pd.to_datetime(silver_pdf['arrival_ts'])

silver_pdf['latency_days'] = (silver_pdf['arrival_ts'] - silver_pdf['report_date']).dt.days

silver_df = spark.createDataFrame(silver_pdf)
silver_df.write.format('delta').mode('overwrite').save('/content/delta/silver_multisource')

print(f'Silver layer: {silver_df.count():,} rows from {N_SOURCES} sources, conformed to canonical schema')
print('\nLatency profile by source (days between report_date and arrival):')
silver_df.groupBy('source_id').agg(F.round(F.avg('latency_days'),2).alias('avg_latency_days')).orderBy('source_id').show(N_SOURCES)

---
## 4 · Gold — unified table with per-source confidence flag

For a given 'as-of' date, a source's most recent data might be 0, 1, or 2 days old depending on its latency profile. The Gold table flags each source's freshness AS OF the query date — so a downstream 'why is this product out of stock' query knows whether it's looking at today's data or 2-day-old data for a given source.

In [ ]:
silver_df.createOrReplaceTempView('silver_multisource')

AS_OF_DATE = '2024-12-20'

gold_df = spark.sql(f'''
    SELECT source_id, product, report_date, inventory_units, units_sold, out_of_stock, latency_days,
        DATEDIFF('{AS_OF_DATE}', report_date) AS days_old_as_of,
        CASE
            WHEN DATEDIFF('{AS_OF_DATE}', report_date) <= 1 THEN 'fresh'
            WHEN DATEDIFF('{AS_OF_DATE}', report_date) <= 3 THEN 'stale'
            ELSE 'expired'
        END AS confidence_flag
    FROM silver_multisource
    WHERE arrival_ts <= '{AS_OF_DATE}'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY source_id, product ORDER BY report_date DESC) = 1
''')

gold_df.write.format('delta').mode('overwrite').save('/content/delta/gold_multisource')

print(f'Gold snapshot as of {AS_OF_DATE} — most recent record per (source, product), with confidence flag:')
gold_df.groupBy('confidence_flag').count().show()
print('\nSample — note some sources show "stale" due to their latency profile:')
gold_df.select('source_id','product','report_date','days_old_as_of','confidence_flag').orderBy('source_id','product').show(8)

---
## 5 · Monitoring — per-source data quality + drift detection (DEMONSTRATED, not just described)

This section directly answers: *how do you monitor model/pipeline health post-deployment, detect drift, and respond?* We define concrete SLA-style thresholds, then show the SRC_07 drift (injected in Section 1 — units→cases unit change) actually firing an alert.

In [ ]:
SLA = dict(
    max_null_rate          = 0.06,
    max_quarantine_rate    = 0.05,
    max_acceptable_latency = 2,
    drift_zscore_threshold = 3.0,
)

print('SLA THRESHOLDS (Layer A — pipeline output guarantees):')
for k,v in SLA.items(): print(f'  {k}: {v}')

print('\n--- DATA QUALITY SLA CHECK ---')
quality_breach = quality_df[(quality_df['null_rate'] > SLA['max_null_rate']) |
                             (quality_df['quarantine_rate'] > SLA['max_quarantine_rate'])]
if len(quality_breach) > 0:
    print('SLA BREACH — sources exceeding quality thresholds:')
    print(quality_breach[['null_rate','quarantine_rate']].round(4))
else:
    print('All sources within data quality SLA.')

print('\n--- FRESHNESS SLA CHECK (as of', AS_OF_DATE, ') ---')
expired = gold_df.filter(F.col('confidence_flag')=='expired').select('source_id','product','days_old_as_of')
expired_count = expired.count()
if expired_count > 0:
    print(f'SLA BREACH — {expired_count} (source,product) pairs exceed {SLA["max_acceptable_latency"]}-day freshness SLA:')
    expired.show()
else:
    print('All sources within freshness SLA.')

In [ ]:
silver_pdf_clean = silver_pdf.dropna(subset=['inventory_units'])

drift_alerts = []
for src in SOURCE_IDS:
    src_data = silver_pdf_clean[silver_pdf_clean['source_id']==src].sort_values('report_date')

    baseline = src_data[src_data['report_date'] < '2024-12-01']
    recent   = src_data[src_data['report_date'] >= '2024-12-15']

    if len(baseline)==0 or len(recent)==0: continue

    base_mean, base_std = baseline['inventory_units'].mean(), baseline['inventory_units'].std()
    recent_mean = recent['inventory_units'].mean()

    zscore = (recent_mean - base_mean) / base_std if base_std>0 else 0

    drift_alerts.append(dict(source_id=src, baseline_mean=round(base_mean,1),
                              recent_mean=round(recent_mean,1), zscore=round(zscore,2),
                              breach=abs(zscore) > SLA['drift_zscore_threshold']))

drift_df = pd.DataFrame(drift_alerts).sort_values('zscore')
print('--- DATA DRIFT CHECK (per-source inventory_units: recent vs. own baseline) ---')
print(drift_df.to_string(index=False))

breached = drift_df[drift_df['breach']]
if len(breached) > 0:
    print(f'\n*** DRIFT ALERT FIRED for: {list(breached["source_id"])} ***')
    print('Diagnosis: inventory_units dropped sharply with no corresponding change in units_sold')
    print('           or out_of_stock rate — consistent with a UNIT-OF-MEASURE change (e.g.')
    print('           the source started reporting in cases instead of units), not a real')
    print('           inventory event. Recommended action: contact source owner to confirm')
    print('           export format before retraining or adjusting downstream models.')
else:
    print('\nNo drift alerts.')

---
## 6 · Visualisation — fleet health dashboard

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Per-source data quality (null + quarantine rate)',
    f'Per-source drift z-score (recent vs. baseline) — threshold = ±{SLA["drift_zscore_threshold"]}'))

fig.add_trace(go.Bar(x=quality_df.index, y=quality_df['null_rate']*100,
    name='Null rate %', marker_color='#2E86C1'), row=1, col=1)
fig.add_trace(go.Bar(x=quality_df.index, y=quality_df['quarantine_rate']*100,
    name='Quarantine rate %', marker_color='#E74C3C'), row=1, col=1)

colors = ['#E74C3C' if b else '#1A5276' for b in drift_df['breach']]
fig.add_trace(go.Bar(x=drift_df['source_id'], y=drift_df['zscore'],
    marker_color=colors, showlegend=False), row=1, col=2)
fig.add_hline(y=SLA['drift_zscore_threshold'], line_dash='dash', line_color='red', row=1, col=2)
fig.add_hline(y=-SLA['drift_zscore_threshold'], line_dash='dash', line_color='red', row=1, col=2)

fig.update_layout(height=400, title_text='Multi-Source Pipeline Health Dashboard', plot_bgcolor='white',
    barmode='group')
fig.update_xaxes(showgrid=False)
fig.show()

print(f'SRC_07 (highlighted in red if breached) is the source with the injected unit-of-measure drift.')

---
## 7 · Production notes — Databricks deployment for 11+ sources

In [ ]:
print('''
PRODUCTION DEPLOYMENT — DATABRICKS, MULTI-SOURCE
==================================================
- Each source = its own Bronze Delta table (catalog.bronze.src_XX) — schema-on-read,
  preserves native format. New source = new table, not a pipeline change.
- Per-source validation runs as a scheduled job per Bronze table — quarantined
  records land in catalog.quarantine.src_XX for review, NOT silently dropped.
- Silver conformance + freshness-aware join: Delta Live Tables (DLT) expectations
  could enforce the canonical schema and flag conformance failures declaratively.
- Gold confidence_flag becomes a first-class column any downstream consumer
  (recommendation model, availability dashboard) can filter/weight on.
- Drift checks run as a scheduled job comparing rolling stats to a versioned
  baseline stored in Unity Catalog — baseline re-established after a confirmed
  (non-drift) change, e.g. a real new product launch.
- SLA breaches (Section 5) -> Databricks job alerts / webhook to ops channel,
  with source_id and specific check that failed — so the right bottler
  relationship owner can be contacted directly.

KEY DESIGN PRINCIPLE: one source's problem is VISIBLE and ISOLATED — it does
not silently degrade the shared Gold table or get averaged away in an
aggregate metric.
''')